# Update fornecedor CAR in Lakehouse

Notebook designed for Microsoft Fabric job type `RunNotebook`.

Expected parameters (executionData.parameters):
- `action` (string): should be `update_fornecedor_car`
- `id_fornecedor` (string)
- `car` (string)
- `requested_by` (string, optional)


In [ ]:
# Parameter cell
action = "update_fornecedor_car"
id_fornecedor = ""
car = ""
requested_by = ""

# Optional controls
target_table = "Fornecedores"
target_schema = "dbo"
dry_run = False


In [ ]:
import json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from delta.tables import DeltaTable

def _clean(value: str) -> str:
    if value is None:
        return ""
    return str(value).strip()

action = _clean(action)
id_fornecedor = _clean(id_fornecedor)
car = _clean(car).upper()
requested_by = _clean(requested_by)
target_table = _clean(target_table) or "Fornecedores"
target_schema = _clean(target_schema) or "dbo"

if action and action != "update_fornecedor_car":
    raise ValueError(f"Invalid action: {action}")

if not id_fornecedor:
    raise ValueError("Parameter id_fornecedor is required")

if not car:
    raise ValueError("Parameter car is required")

database_name = spark.catalog.currentDatabase()
candidates = [
    f"{database_name}.{target_schema}.{target_table}",
    f"{database_name}.{target_table}",
    f"{target_schema}.{target_table}",
    target_table,
]

target_identifier = None
for name in candidates:
    try:
        spark.table(name).limit(1).collect()
        target_identifier = name
        break
    except Exception:
        continue

if not target_identifier:
    raise RuntimeError(
        f"Could not resolve target table. Tried: {candidates}"
    )

base_df = spark.table(target_identifier)
exists = base_df.where(F.col("id_fornecedor") == id_fornecedor).limit(1).count() > 0
if not exists:
    raise RuntimeError(f"Fornecedor not found: {id_fornecedor}")

if not dry_run:
    updates_df = spark.createDataFrame(
        [(id_fornecedor, car)],
        ["id_fornecedor", "car"],
    )
    target = DeltaTable.forName(spark, target_identifier)
    target.alias("t").merge(
        updates_df.alias("s"),
        "t.id_fornecedor = s.id_fornecedor",
    ).whenMatchedUpdate(
        set={
            "car": "s.car",
            "updated_at": "current_timestamp()",
        }
    ).execute()

updated_row = (
    spark.table(target_identifier)
    .where(F.col("id_fornecedor") == id_fornecedor)
    .select("id_fornecedor", "cpf_cnpj", "nome", "car", "updated_at")
    .limit(1)
    .collect()[0]
)

result = {
    "ok": True,
    "action": "update_fornecedor_car",
    "id_fornecedor": updated_row["id_fornecedor"],
    "cpf_cnpj": updated_row["cpf_cnpj"],
    "nome": updated_row["nome"],
    "car": updated_row["car"],
    "updated_at": str(updated_row["updated_at"]),
    "requested_by": requested_by or None,
    "dry_run": bool(dry_run),
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
}

result_json = json.dumps(result, ensure_ascii=False)
print(result_json)

try:
    import notebookutils
    notebookutils.notebook.exit(result_json)
except Exception:
    pass
